This notebook starts by building the TFIDF for the 3 Disneyland locations, before testing NMF modeling. This notebook iwll use loops of the same TFIDF and NMF code to ensure consistency across California, Paris, and Hong Kong segments of the dataset. 

In [55]:
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [57]:
# load the processed dataframe (only stored locally due to 100mb+ size)
disneyland_df = pd.read_csv('../data/processed/disneyland_df_processed.csv')

print("shape:", disneyland_df.shape)
print("\ncolumns:")
print(disneyland_df.columns.tolist())

shape: (42479, 10)

columns:
['Review_ID', 'Rating', 'Year_Month', 'Reviewer_Location', 'Review_Text', 'Branch', 'review_cleaned', 'clean_review_wo_stopwords', 'tokenized_texts', 'cleaned_texts']


In [59]:
# tokenized_texts may come back from CSV as strings instead of real lists
if isinstance(disneyland_df['tokenized_texts'].iloc[0], str):
    disneyland_df['tokenized_texts'] = disneyland_df['tokenized_texts'].apply(ast.literal_eval)

print(type(disneyland_df['tokenized_texts'].iloc[0]))

<class 'list'>


In [61]:
print(disneyland_df['Branch'].value_counts())

Branch
Disneyland_California    19287
Disneyland_Paris         13598
Disneyland_HongKong       9594
Name: count, dtype: int64


In [63]:
# split the dataframe into the 3 Disneyland branches
california_reviews = disneyland_df[disneyland_df['Branch'] == 'Disneyland_California'].copy()
paris_reviews = disneyland_df[disneyland_df['Branch'] == 'Disneyland_Paris'].copy()
hong_kong_reviews = disneyland_df[disneyland_df['Branch'] == 'Disneyland_HongKong'].copy()

# reset index to keep each group clean and easier to work with
california_reviews.reset_index(drop=True, inplace=True)
paris_reviews.reset_index(drop=True, inplace=True)
hong_kong_reviews.reset_index(drop=True, inplace=True)

# check sizes
print("California shape:", california_reviews.shape)
print("Paris shape:", paris_reviews.shape)
print("Hong Kong shape:", hong_kong_reviews.shape)

California shape: (19287, 10)
Paris shape: (13598, 10)
Hong Kong shape: (9594, 10)


In [65]:
print("Missing cleaned_texts values:")
print("California:", california_reviews['cleaned_texts'].isna().sum())
print("Paris:", paris_reviews['cleaned_texts'].isna().sum())
print("Hong Kong:", hong_kong_reviews['cleaned_texts'].isna().sum())

Missing cleaned_texts values:
California: 0
Paris: 0
Hong Kong: 0


In [67]:
def build_tfidf_matrix(cleaned_texts):
    """
    Build a TF-IDF matrix from cleaned review text.
    This follows the same general setup used in the example notebook.
    """
    
    vectorizer = TfidfVectorizer(
        stop_words='english',
        max_features=10_000,
        min_df=5,          # word must appear in at least 5 docs
        max_df=0.80,       # ignore words in more than 80% of docs
        ngram_range=(1, 2) # include unigrams and bigrams
    )
    
    tfidf_matrix = vectorizer.fit_transform(cleaned_texts)
    feature_names = vectorizer.get_feature_names_out()
    
    return vectorizer, tfidf_matrix, feature_names

In [73]:
# California TF-IDF matrix
california_vectorizer, california_tfidf, california_features = build_tfidf_matrix(
    california_reviews['cleaned_texts']
)

print(f'California TF-IDF matrix shape: {california_tfidf.shape}')
print(f'  → {california_tfidf.shape[0]:,} documents × {california_tfidf.shape[1]:,} features')
print()

print('Example — first California doc TF-IDF (top 8 tokens by weight):')
row = california_tfidf[0].toarray()[0]
top_idx = row.argsort()[::-1][:8]
for idx in top_idx:
    print(f'  {california_features[idx]:25s}: {row[idx]:.4f}')

California TFIDF matrix shape: (19287, 10000)
  → 19,287 documents × 10,000 features

Example — first California doc TFIDF (top 8 tokens by weight):
  kid spend                : 0.3234
  especially young         : 0.3186
  different attraction     : 0.3186
  priceless                : 0.2693
  regret                   : 0.2657
  feeling                  : 0.2534
  forever                  : 0.2402
  spend money              : 0.2386


In [75]:
# Paris TF-IDF matrix
paris_vectorizer, paris_tfidf, paris_features = build_tfidf_matrix(
    paris_reviews['cleaned_texts']
)

print(f'Paris TF-IDF matrix shape: {paris_tfidf.shape}')
print(f'  → {paris_tfidf.shape[0]:,} documents × {paris_tfidf.shape[1]:,} features')
print()

print('Example — first Paris doc TF-IDF (top 8 tokens by weight):')
row = paris_tfidf[0].toarray()[0]
top_idx = row.argsort()[::-1][:8]
for idx in top_idx:
    print(f'  {paris_features[idx]:25s}: {row[idx]:.4f}')

Paris TFIDF matrix shape: (13598, 10000)
  → 13,598 documents × 10,000 features

Example — first Paris doc TFIDF (top 8 tokens by weight):
  fast track               : 0.2985
  track                    : 0.2877
  visit fast               : 0.2411
  firework world           : 0.2390
  hotel resort             : 0.2334
  hongkong                 : 0.2334
  forward visit            : 0.2188
  visit big                : 0.2178


In [79]:
# Hong Kong TF-IDF matrix
hong_kong_vectorizer, hong_kong_tfidf, hong_kong_features = build_tfidf_matrix(
    hong_kong_reviews['cleaned_texts']
)

print(f'Hong Kong TF-IDF matrix shape: {hong_kong_tfidf.shape}')
print(f'  → {hong_kong_tfidf.shape[0]:,} documents × {hong_kong_tfidf.shape[1]:,} features')
print()

print('Example — first Hong Kong doc TF-IDF (top 8 tokens by weight):')
row = hong_kong_tfidf[0].toarray()[0]
top_idx = row.argsort()[::-1][:8]
for idx in top_idx:
    print(f'  {hong_kong_features[idx]:25s}: {row[idx]:.4f}')

Hong Kong TFIDF matrix shape: (9594, 10000)
  → 9,594 documents × 10,000 features

Example — first Hong Kong doc TFIDF (top 8 tokens by weight):
  fairly                   : 0.4644
  busy queue               : 0.2987
  walk main                : 0.2710
  familiar                 : 0.2557
  layout                   : 0.2497
  rid small                : 0.2407
  fabulous                 : 0.2212
  relatively               : 0.2155


In [81]:
# tokenized texts for coherence scoring
california_tokens = california_reviews['tokenized_texts'].tolist()
paris_tokens = paris_reviews['tokenized_texts'].tolist()
hong_kong_tokens = hong_kong_reviews['tokenized_texts'].tolist()

# gensim dictionaries for coherence
california_dictionary = Dictionary(california_tokens)
paris_dictionary = Dictionary(paris_tokens)
hong_kong_dictionary = Dictionary(hong_kong_tokens)

In [83]:
# keep all branch-specific objects together for easier looping later
group_data = {
    'California': {
        'df': california_reviews,
        'tfidf_matrix': california_tfidf,
        'features': california_features,
        'tokens': california_tokens,
        'dictionary': california_dictionary
    },
    'Paris': {
        'df': paris_reviews,
        'tfidf_matrix': paris_tfidf,
        'features': paris_features,
        'tokens': paris_tokens,
        'dictionary': paris_dictionary
    },
    'Hong Kong': {
        'df': hong_kong_reviews,
        'tfidf_matrix': hong_kong_tfidf,
        'features': hong_kong_features,
        'tokens': hong_kong_tokens,
        'dictionary': hong_kong_dictionary
    }
}

In [87]:
for group_name, group_info in group_data.items():
    print(group_name)
    print("TF-IDF shape:", group_info['tfidf_matrix'].shape)
    print("Vocabulary size:", len(group_info['features']))
    print()

California
TFIDF shape: (19287, 10000)
Vocabulary size: 10000

Paris
TFIDF shape: (13598, 10000)
Vocabulary size: 10000

Hong Kong
TFIDF shape: (9594, 10000)
Vocabulary size: 10000



Now that TFIDF matrices are complete for all 3 Disneyland segments, we will run NMF model testing based on the required testing k values of 5, 8, 10, 12, and 15. Our NMF model will use: 

- init = nndsvda
- random_state = 42
- max_iter = 500
- l1_ratio = 0.5

In [90]:
# topic counts required for the assignment
k_values = [5, 8, 10, 12, 15]
print("k values to test:", k_values)

k values to test: [5, 8, 10, 12, 15]


In [92]:
# tokenized texts for each group
california_tokens = california_reviews['tokenized_texts'].tolist()
paris_tokens = paris_reviews['tokenized_texts'].tolist()
hong_kong_tokens = hong_kong_reviews['tokenized_texts'].tolist()

# dictionaries for coherence scoring
california_dictionary = Dictionary(california_tokens)
paris_dictionary = Dictionary(paris_tokens)
hong_kong_dictionary = Dictionary(hong_kong_tokens)

In [94]:
# keep all branch-specific objects together so later code is easier to manage
group_data = {
    'California': {
        'df': california_reviews,
        'tfidf_matrix': california_tfidf,
        'features': california_features,
        'tokens': california_tokens,
        'dictionary': california_dictionary
    },
    'Paris': {
        'df': paris_reviews,
        'tfidf_matrix': paris_tfidf,
        'features': paris_features,
        'tokens': paris_tokens,
        'dictionary': paris_dictionary
    },
    'Hong Kong': {
        'df': hong_kong_reviews,
        'tfidf_matrix': hong_kong_tfidf,
        'features': hong_kong_features,
        'tokens': hong_kong_tokens,
        'dictionary': hong_kong_dictionary
    }
}

In [96]:
def get_top_words_per_topic(model, feature_names, top_n=10):
    """
    Get the top words for each topic from a fitted NMF model.
    """
    
    topic_words = []
    
    for topic in model.components_:
        top_idx = topic.argsort()[::-1][:top_n]
        top_words = [feature_names[i] for i in top_idx]
        topic_words.append(top_words)
    
    return topic_words

In [98]:
def compute_nmf_coherence(topic_words, tokenized_texts, dictionary):
    """
    Compute c_v coherence score for NMF topics.
    """
    
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    
    return coherence_model.get_coherence()

In [100]:
def evaluate_nmf_models(tfidf_matrix, feature_names, tokenized_texts, dictionary, k_values):
    """
    Fit NMF models for multiple k values and store the main outputs.
    
    For each k, this function saves:
    - fitted NMF model
    - document-topic matrix (W)
    - topic-term matrix (H)
    - reconstruction error
    - coherence score
    - top words per topic
    """
    
    results = []
    
    for k in k_values:
        print(f'Running NMF for k = {k}...')
        
        # NMF setup follows the professor's example as closely as possible
        nmf_model = NMF(
            n_components=k,
            random_state=42,
            init='nndsvda',
            max_iter=500,
            l1_ratio=0.5
        )
        
        # W matrix = documents x topics
        nmf_matrix = nmf_model.fit_transform(tfidf_matrix)
        
        # H matrix = topics x vocabulary
        topic_term_matrix = nmf_model.components_
        
        print(f'NMF W matrix shape: {nmf_matrix.shape}  (docs × topics)')
        print(f'NMF H matrix shape: {topic_term_matrix.shape}  (topics × vocab)')
        print(f'Reconstruction Error: {nmf_model.reconstruction_err_:.4f}')
        print('  → Lower = TF-IDF matrix is better approximated by W × H')
        print()
        
        # get top words for coherence and later interpretation
        topic_words = get_top_words_per_topic(nmf_model, feature_names, top_n=10)
        
        # compute coherence score
        coherence_score = compute_nmf_coherence(
            topic_words=topic_words,
            tokenized_texts=tokenized_texts,
            dictionary=dictionary
        )
        
        print(f'Coherence Score: {coherence_score:.4f}')
        print('-' * 60)
        
        results.append({
            'k': k,
            'model': nmf_model,
            'document_topic_matrix': nmf_matrix,
            'topic_term_matrix': topic_term_matrix,
            'reconstruction_error': nmf_model.reconstruction_err_,
            'coherence_score': coherence_score,
            'topic_words': topic_words
        })
    
    return results

In [102]:
# run NMF models for California
california_nmf_results = evaluate_nmf_models(
    tfidf_matrix=group_data['California']['tfidf_matrix'],
    feature_names=group_data['California']['features'],
    tokenized_texts=group_data['California']['tokens'],
    dictionary=group_data['California']['dictionary'],
    k_values=k_values
)

Running NMF for k = 5...
NMF W matrix shape: (19287, 5)  (docs × topics)
NMF H matrix shape: (5, 10000)  (topics × vocab)
Reconstruction Error: 135.5122
  → Lower = TF-IDF matrix is better approximated by W × H

Coherence Score: 0.5960
------------------------------------------------------------
Running NMF for k = 8...
NMF W matrix shape: (19287, 8)  (docs × topics)
NMF H matrix shape: (8, 10000)  (topics × vocab)
Reconstruction Error: 134.8345
  → Lower = TF-IDF matrix is better approximated by W × H

Coherence Score: 0.5791
------------------------------------------------------------
Running NMF for k = 10...
NMF W matrix shape: (19287, 10)  (docs × topics)
NMF H matrix shape: (10, 10000)  (topics × vocab)
Reconstruction Error: 134.4172
  → Lower = TF-IDF matrix is better approximated by W × H

Coherence Score: 0.6071
------------------------------------------------------------
Running NMF for k = 12...
NMF W matrix shape: (19287, 12)  (docs × topics)
NMF H matrix shape: (12, 10000)

In [103]:
# run NMF models for Paris
paris_nmf_results = evaluate_nmf_models(
    tfidf_matrix=group_data['Paris']['tfidf_matrix'],
    feature_names=group_data['Paris']['features'],
    tokenized_texts=group_data['Paris']['tokens'],
    dictionary=group_data['Paris']['dictionary'],
    k_values=k_values
)

Running NMF for k = 5...
NMF W matrix shape: (13598, 5)  (docs × topics)
NMF H matrix shape: (5, 10000)  (topics × vocab)
Reconstruction Error: 113.6496
  → Lower = TF-IDF matrix is better approximated by W × H

Coherence Score: 0.5603
------------------------------------------------------------
Running NMF for k = 8...
NMF W matrix shape: (13598, 8)  (docs × topics)
NMF H matrix shape: (8, 10000)  (topics × vocab)
Reconstruction Error: 113.0956
  → Lower = TF-IDF matrix is better approximated by W × H

Coherence Score: 0.5835
------------------------------------------------------------
Running NMF for k = 10...
NMF W matrix shape: (13598, 10)  (docs × topics)
NMF H matrix shape: (10, 10000)  (topics × vocab)
Reconstruction Error: 112.7873
  → Lower = TF-IDF matrix is better approximated by W × H

Coherence Score: 0.5679
------------------------------------------------------------
Running NMF for k = 12...
NMF W matrix shape: (13598, 12)  (docs × topics)
NMF H matrix shape: (12, 10000)

In [104]:
# run NMF models for Hong Kong
hong_kong_nmf_results = evaluate_nmf_models(
    tfidf_matrix=group_data['Hong Kong']['tfidf_matrix'],
    feature_names=group_data['Hong Kong']['features'],
    tokenized_texts=group_data['Hong Kong']['tokens'],
    dictionary=group_data['Hong Kong']['dictionary'],
    k_values=k_values
)

Running NMF for k = 5...
NMF W matrix shape: (9594, 5)  (docs × topics)
NMF H matrix shape: (5, 10000)  (topics × vocab)
Reconstruction Error: 95.9149
  → Lower = TF-IDF matrix is better approximated by W × H

Coherence Score: 0.5603
------------------------------------------------------------
Running NMF for k = 8...
NMF W matrix shape: (9594, 8)  (docs × topics)
NMF H matrix shape: (8, 10000)  (topics × vocab)
Reconstruction Error: 95.4695
  → Lower = TF-IDF matrix is better approximated by W × H

Coherence Score: 0.5519
------------------------------------------------------------
Running NMF for k = 10...
NMF W matrix shape: (9594, 10)  (docs × topics)
NMF H matrix shape: (10, 10000)  (topics × vocab)
Reconstruction Error: 95.1972
  → Lower = TF-IDF matrix is better approximated by W × H

Coherence Score: 0.5789
------------------------------------------------------------
Running NMF for k = 12...
NMF W matrix shape: (9594, 12)  (docs × topics)
NMF H matrix shape: (12, 10000)  (topi

In [105]:
# keep all results in one place for easier comparison later
nmf_results = {
    'California': california_nmf_results,
    'Paris': paris_nmf_results,
    'Hong Kong': hong_kong_nmf_results
}

In [106]:
# create a summary table for reconstruction error and coherence score
summary_rows = []

for group_name, results_list in nmf_results.items():
    for result in results_list:
        summary_rows.append({
            'Group': group_name,
            'k': result['k'],
            'Reconstruction Error': result['reconstruction_error'],
            'Coherence Score': result['coherence_score']
        })

nmf_summary_df = pd.DataFrame(summary_rows)
nmf_summary_df = nmf_summary_df.sort_values(by=['Group', 'k']).reset_index(drop=True)

nmf_summary_df

,Group,k,Reconstruction Error,Coherence Score
0,California,5,135.512164,0.596034
1,California,8,134.834533,0.579069
2,California,10,134.417153,0.607148
3,California,12,134.047446,0.592497
4,California,15,133.516262,0.609992
5,Hong Kong,5,95.914853,0.560312
6,Hong Kong,8,95.469494,0.551895
7,Hong Kong,10,95.197171,0.578915
8,Hong Kong,12,94.927608,0.609697
9,Hong Kong,15,94.566366,0.624913


In [107]:
# quick example: preview California topics when k = 5
for result in california_nmf_results:
    if result['k'] == 5:
        print('California, k = 5')
        print()
        for i, words in enumerate(result['topic_words'], start=1):
            print(f'Topic {i}: {", ".join(words)}')

California, k = 5

Topic 1: line, wait, ride, long, hour, people, minute, crowd, long line, ticket
Topic 2: happy, place, earth, place earth, happy place, truly happy, truly, magical, great place, staff
Topic 3: pass, fast, fast pass, use, use fast, ride, early, hopper, pass ride, app
Topic 4: mountain, ride, space, space mountain, jones, indiana, indiana jones, world, pirate, splash
Topic 5: great, love, year, visit, kid, fun, old, family, place, experience
